# Bundesliga Standings and Expected-Performance Snapshot

This notebook retrieves the live **2026/27 Bundesliga standings** from SofaScore and creates a compact, validated JSON snapshot. When The Analyst expected-performance dataset is no more than **seven elapsed days (604,800 seconds)** old, the notebook can enrich confirmed teams with expected position, points, goals scored, and goals conceded, plus locally calculated actual-versus-expected differences.

The resulting file is named `bundesliga_table_YYYY-MM-DD_HH-MM-SS.json` and is written to `outputs/derived/bundesliga_snapshots`. Expected data that is stale, unavailable, malformed, or not confirmed is never silently used.

## Requirements

**Required local file:**
- `outputs/sofascore/reference/bundesliga_teams.json` — the authoritative mapping of SofaScore team IDs to Bundesliga clubs, generated by the team-reference notebook.

**External API sources:**
- SofaScore Bundesliga standings API
- The Analyst expected-points API

**Reference-only files used during development:**
- `standings api call.txt`
- `expected points api call.txt`

The two `.txt` files are structural references only and are **not required during normal execution**. The notebook always retrieves live API data at runtime.

**Software requirements:**
- Internet access
- Google Chrome installed
- Python packages: `pandas`, `undetected-chromedriver`, `beautifulsoup4`, and `selenium`
- The browser is explicitly started with `uc.Chrome(version_main=150)`.

Install missing packages in the active Jupyter kernel with:

```python
%pip install pandas undetected-chromedriver beautifulsoup4 selenium
```

## Freshness and interactive behavior

Expected data is eligible only when `current_datetime - lastUpdated <= 7 days`, calculated from actual elapsed seconds rather than calendar dates. A future timestamp produces a warning but remains eligible. Stale or invalid expected data is excluded, while the SofaScore standings snapshot is still generated.

Every proposed SofaScore–Analyst team pairing requires explicit confirmation through a case-insensitive `y/n` `input()` prompt—even exact-name, alias, and team-code proposals. A rejected proposal may be followed by the next plausible candidate. The notebook does not check every possible Cartesian combination and does not ask matching questions when expected data is stale or unavailable. If interactive input is unavailable or no mappings are confirmed, a standings-only snapshot is produced.

> **Directory note:** paths are resolved through `project_paths.py`. Start Jupyter from the project root or open the notebook from within the project tree.


In [1]:
# Resolve the project root and import authoritative data locations.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    DERIVED_BUNDESLIGA_SNAPSHOTS_DIR,
    PROJECT_ROOT,
    SOFASCORE_REFERENCE_DIR,
    ensure_directory,
)


## 1. Imports and configuration

The source URLs, season, Chrome major version, freshness threshold, and browser timeouts are fixed here.


In [2]:
# Import the libraries required by this notebook step.
import difflib
import json
import math
import os
import re
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# Handle expected failures with a clear, actionable message.
try:
    import pandas as pd
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from IPython.display import display
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        "Required packages are missing. Install them in this Jupyter kernel with: "
        "%pip install pandas undetected-chromedriver beautifulsoup4 selenium"
    ) from exc

# Set workflow configuration value: CHROME_MAJOR_VERSION.
CHROME_MAJOR_VERSION = 150
# Set workflow configuration value: COMPETITION.
COMPETITION = "Bundesliga"
# Set workflow configuration value: SEASON.
SEASON = "2026/27"
# Set workflow configuration value: EXPECTED_TEAM_COUNT.
EXPECTED_TEAM_COUNT = 18
# Set workflow configuration value: MAX_EXPECTED_AGE_SECONDS.
MAX_EXPECTED_AGE_SECONDS = 7 * 24 * 60 * 60
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 45
# Set workflow configuration value: PRE_WAIT_TIMEOUT_SECONDS.
PRE_WAIT_TIMEOUT_SECONDS = 30
# Set workflow configuration value: SIMILARITY_PROMPT_THRESHOLD.
SIMILARITY_PROMPT_THRESHOLD = 0.70

# Set workflow configuration value: STANDINGS_URL.
STANDINGS_URL = (
    "https://www.sofascore.com/api/v1/unique-tournament/35/"
    "season/97464/standings/total"
)
# Set workflow configuration value: EXPECTED_POINTS_URL.
EXPECTED_POINTS_URL = (
    "https://dataviz.theanalyst.com/project-data/soccer/"
    "2bchmrj23l9u42d68ntcekob8/expected-points.json"
)



## 2. Resolve project data locations

The shared path module supplies the team-reference input and derived snapshot output locations without machine-specific paths.


In [3]:
# Run this self-contained workflow step using the prepared inputs.
base_directory = PROJECT_ROOT
teams_path = SOFASCORE_REFERENCE_DIR / "bundesliga_teams.json"

print(f"Project root: {base_directory}")
print(f"Required team mapping: {teams_path}")


Project root: C:\kickbase project
Required team mapping: C:\kickbase project\outputs\sofascore\reference\bundesliga_teams.json


## 3. Load the authoritative Bundesliga team mapping

The mapping defines the exact team-ID set expected in the snapshot. Team IDs—not names—are the primary identifiers.


In [4]:
# Validate the input before continuing with later processing.
if not teams_path.is_file():
    raise FileNotFoundError(
        f"Required team mapping was not found: {teams_path}. Run the team-reference notebook first."
    )

# Handle expected failures with a clear, actionable message.
try:
    raw_teams = json.loads(teams_path.read_text(encoding="utf-8"))
except UnicodeDecodeError as exc:
    raise ValueError(f"Required team mapping is not valid UTF-8: {teams_path}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Required team mapping is invalid JSON at line {exc.lineno}, "
        f"column {exc.colno}: {teams_path}"
    ) from exc
except OSError as exc:
    raise OSError(f"Could not read required team mapping {teams_path}: {exc}") from exc

# Validate the input before continuing with later processing.
if not isinstance(raw_teams, dict):
    raise ValueError("bundesliga_teams.json must be an object keyed by team ID.")
# Validate the input before continuing with later processing.
if len(raw_teams) != EXPECTED_TEAM_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_TEAM_COUNT} Bundesliga teams, found {len(raw_teams)}."
    )

teams_by_id: dict[int, dict[str, Any]] = {}
# Process each available item while preserving the current workflow state.
for team_key, team_record in raw_teams.items():
    # Validate the input before continuing with later processing.
    if not isinstance(team_record, dict):
        raise ValueError(f"Team entry {team_key!r} must be a JSON object.")

    team_id = team_record.get("team_id")
    team_name = team_record.get("team")
    # Validate the input before continuing with later processing.
    if not isinstance(team_id, int) or isinstance(team_id, bool) or team_id <= 0:
        raise ValueError(f"Team entry {team_key!r} has an invalid team_id.")
    # Validate the input before continuing with later processing.
    if str(team_id) != str(team_key):
        raise ValueError(
            f"Team key {team_key!r} does not match its team_id {team_id!r}."
        )
    # Validate the input before continuing with later processing.
    if not isinstance(team_name, str) or not team_name.strip():
        raise ValueError(f"Team entry {team_key!r} has an invalid team name.")
    # Validate the input before continuing with later processing.
    if team_id in teams_by_id:
        raise ValueError(f"Duplicate team ID in mapping: {team_id}.")

    teams_by_id[team_id] = {"team_id": team_id, "team": team_name.strip()}

print(f"Loaded and validated {len(teams_by_id)} authoritative teams.")


Loaded and validated 18 authoritative teams.


## 4. Shared normalization and validation helpers

Name normalization is used only for comparison. Original source spellings remain unchanged in prompts and output.


In [5]:
# Check whether number for reuse in the workflow.
def is_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


# Handle integer for reuse in the workflow.
def require_integer(value: Any, field_name: str, *, minimum: int | None = None) -> int:
    # Validate the input before continuing with later processing.
    if not isinstance(value, int) or isinstance(value, bool):
        raise ValueError(f"{field_name} must be an integer; received {value!r}.")
    # Validate the input before continuing with later processing.
    if minimum is not None and value < minimum:
        raise ValueError(f"{field_name} must be at least {minimum}; received {value}.")
    return value


# Normalize name for reuse in the workflow.
def normalize_name(value: str) -> str:
    normalized = unicodedata.normalize("NFKC", value).casefold()
    decomposed = unicodedata.normalize("NFKD", normalized)
    without_marks = "".join(
        character for character in decomposed if not unicodedata.combining(character)
    )
    return re.sub(r"[^a-z0-9]+", " ", without_marks).strip()


# Normalize code for reuse in the workflow.
def normalize_code(value: Any) -> str | None:
    if not isinstance(value, str) or not value.strip():
        return None
    return value.strip().upper()


# Set workflow configuration value: ALIAS_GROUPS.
ALIAS_GROUPS = [
    ["FC Bayern München", "Bayern München"],
    ["Bayer 04 Leverkusen", "Bayer Leverkusen"],
    ["Borussia M'gladbach", "Borussia VfL Mönchengladbach"],
    ["RB Leipzig", "RasenBallsport Leipzig"],
    ["Borussia Dortmund", "BV Borussia 09 Dortmund"],
    ["VfB Stuttgart", "VfB Stuttgart 1893"],
    ["TSG Hoffenheim", "TSG 1899 Hoffenheim"],
]

# Set workflow configuration value: ALIAS_CANONICAL.
ALIAS_CANONICAL: dict[str, str] = {}
# Process each available item while preserving the current workflow state.
for alias_group in ALIAS_GROUPS:
    canonical = normalize_name(alias_group[0])
    # Process each available item while preserving the current workflow state.
    for alias in alias_group:
        ALIAS_CANONICAL[normalize_name(alias)] = canonical


# Handle name for reuse in the workflow.
def canonical_name(value: str) -> str:
    normalized = normalize_name(value)
    return ALIAS_CANONICAL.get(normalized, normalized)


# Handle are compatible for reuse in the workflow.
def names_are_compatible(left: str, right: str) -> bool:
    return normalize_name(left) == normalize_name(right) or canonical_name(left) == canonical_name(right)


## 5. Browser/API helper and live retrieval

One reusable undetected-Chrome session loads both JSON endpoints. SofaScore is mandatory; Analyst failure is handled as optional-enrichment failure.


In [6]:
# Define Api Retrieval Error to keep related behaviour explicit.
class ApiRetrievalError(RuntimeError):
    """Raised when Chrome cannot return a usable JSON API response."""


# Retrieve json with chrome for reuse in the workflow.
def fetch_json_with_chrome(driver: Any, url: str, source_name: str) -> Any:
    # Handle expected failures with a clear, actionable message.
    try:
        driver.get(url)
    except TimeoutException as exc:
        raise ApiRetrievalError(
            f"{source_name} timed out after {PAGE_LOAD_TIMEOUT_SECONDS} seconds: {url}"
        ) from exc
    except WebDriverException as exc:
        raise ApiRetrievalError(f"Chrome could not load {source_name}: {url}") from exc

    # Handle expected failures with a clear, actionable message.
    try:
        WebDriverWait(driver, PRE_WAIT_TIMEOUT_SECONDS).until(
            EC.presence_of_element_located((By.TAG_NAME, "pre"))
        )
    except (TimeoutException, WebDriverException) as exc:
        raise ApiRetrievalError(
            f"Could not locate the JSON <pre> element for {source_name}: {url}"
        ) from exc

    soup = BeautifulSoup(driver.page_source, "html.parser")
    pre_tag = soup.find("pre")
    # Validate the input before continuing with later processing.
    if pre_tag is None:
        raise ApiRetrievalError(f"Rendered {source_name} response contains no <pre>: {url}")

    response_text = pre_tag.get_text().strip()
    # Validate the input before continuing with later processing.
    if not response_text:
        raise ApiRetrievalError(f"Rendered {source_name} JSON response is empty: {url}")

    # Handle expected failures with a clear, actionable message.
    try:
        return json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise ApiRetrievalError(
            f"{source_name} returned invalid JSON at line {exc.lineno}, column {exc.colno}."
        ) from exc


In [7]:
driver = None
analyst_payload = None
analyst_retrieval_error: Exception | None = None

# Handle expected failures with a clear, actionable message.
try:
    # Handle expected failures with a clear, actionable message.
    try:
        driver = uc.Chrome(version_main=150)
        driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
    except Exception as exc:
        raise RuntimeError(
            "Could not initialize undetected Chrome with major version 150. "
            "Confirm that Google Chrome is installed and compatible."
        ) from exc

    print("Retrieving SofaScore Bundesliga standings...")
    sofascore_payload = fetch_json_with_chrome(
        driver, STANDINGS_URL, "SofaScore standings"
    )
    capture_datetime = datetime.now().astimezone()
    print("SofaScore standings retrieved successfully.")

    print("Retrieving The Analyst expected-performance data...")
    # Handle expected failures with a clear, actionable message.
    try:
        analyst_payload = fetch_json_with_chrome(
            driver, EXPECTED_POINTS_URL, "The Analyst expected-performance data"
        )
        print("The Analyst response retrieved successfully.")
    except Exception as exc:
        analyst_retrieval_error = exc
        print(f"Warning: Analyst enrichment is unavailable: {exc}")
finally:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
        except Exception as exc:
            print(f"Warning: Chrome did not close cleanly: {exc}")


Retrieving SofaScore Bundesliga standings...
SofaScore standings retrieved successfully.
Retrieving The Analyst expected-performance data...
The Analyst response retrieved successfully.


## 6. Parse and clean the current standings

Only analytical fields are retained. Temporary team codes support matching but never enter the final JSON.


In [8]:
# Parse and validate standings for reuse in the workflow.
def parse_standings(payload: Any) -> tuple[dict[int, dict[str, Any]], dict[int, str | None]]:
    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise ValueError("SofaScore response must be a JSON object.")

    standings = payload.get("standings")
    # Validate the input before continuing with later processing.
    if not isinstance(standings, list):
        raise ValueError("SofaScore response has no valid standings list.")

    total_tables = [table for table in standings if isinstance(table, dict) and table.get("type") == "total"]
    # Validate the input before continuing with later processing.
    if len(total_tables) != 1:
        raise ValueError(
            f"Expected exactly one SofaScore total standings table, found {len(total_tables)}."
        )

    rows = total_tables[0].get("rows")
    # Validate the input before continuing with later processing.
    if not isinstance(rows, list):
        raise ValueError("SofaScore total standings has no valid rows list.")

    clean_records: dict[int, dict[str, Any]] = {}
    team_codes: dict[int, str | None] = {}

    # Process each available item while preserving the current workflow state.
    for row_index, row in enumerate(rows):
        # Validate the input before continuing with later processing.
        if not isinstance(row, dict):
            raise ValueError(f"SofaScore row {row_index} is not an object.")
        team = row.get("team")
        # Validate the input before continuing with later processing.
        if not isinstance(team, dict):
            raise ValueError(f"SofaScore row {row_index} has no valid team object.")

        team_id = require_integer(team.get("id"), f"rows[{row_index}].team.id", minimum=1)
        team_name = team.get("name")
        # Validate the input before continuing with later processing.
        if not isinstance(team_name, str) or not team_name.strip():
            raise ValueError(f"SofaScore team {team_id} has no valid name.")
        # Validate the input before continuing with later processing.
        if team_id in clean_records:
            raise ValueError(f"Duplicate SofaScore team ID: {team_id}.")
        # Validate the input before continuing with later processing.
        if team_id not in teams_by_id:
            raise ValueError(f"Unexpected SofaScore team ID not present in mapping: {team_id}.")
        # Validate the input before continuing with later processing.
        if not names_are_compatible(team_name, teams_by_id[team_id]["team"]):
            raise ValueError(
                f"SofaScore team ID {team_id} has incompatible names: "
                f"{team_name!r} versus mapping {teams_by_id[team_id]['team']!r}."
            )

        position = require_integer(row.get("position"), f"{team_name}.position", minimum=1)
        matches = require_integer(row.get("matches"), f"{team_name}.matches", minimum=0)
        wins = require_integer(row.get("wins"), f"{team_name}.wins", minimum=0)
        draws = require_integer(row.get("draws"), f"{team_name}.draws", minimum=0)
        losses = require_integer(row.get("losses"), f"{team_name}.losses", minimum=0)
        goals_for = require_integer(row.get("scoresFor"), f"{team_name}.scoresFor", minimum=0)
        goals_against = require_integer(
            row.get("scoresAgainst"), f"{team_name}.scoresAgainst", minimum=0
        )
        points = row.get("points")
        # Validate the input before continuing with later processing.
        if not is_number(points):
            raise ValueError(f"{team_name}.points must be numeric; received {points!r}.")

        clean_records[team_id] = {
            "team_id": team_id,
            "team": team_name.strip(),
            "position": position,
            "matches": matches,
            "wins": wins,
            "draws": draws,
            "losses": losses,
            "goals_for": goals_for,
            "goals_against": goals_against,
            "goal_difference": goals_for - goals_against,
            "points": points,
        }
        team_codes[team_id] = normalize_code(team.get("nameCode"))

    return clean_records, team_codes


standings_by_id, sofascore_codes = parse_standings(sofascore_payload)
# Validate the input before continuing with later processing.
if set(standings_by_id) != set(teams_by_id):
    missing = sorted(set(teams_by_id) - set(standings_by_id))
    unexpected = sorted(set(standings_by_id) - set(teams_by_id))
    raise ValueError(
        f"Standings IDs do not match the authoritative mapping. Missing={missing}; "
        f"unexpected={unexpected}."
    )

print(f"Parsed {len(standings_by_id)} clean standings records.")


Parsed 18 clean standings records.


## 7. Check Analyst structure and freshness

The freshness gate runs before matching, so stale data cannot trigger irrelevant confirmation prompts.


In [9]:
# Parse and validate last updated for reuse in the workflow.
def parse_last_updated(value: Any) -> datetime:
    # Validate the input before continuing with later processing.
    if not isinstance(value, str) or not value.strip():
        raise ValueError("The Analyst lastUpdated value must be a non-empty string.")

    timestamp_text = value.strip()
    if timestamp_text.endswith(("Z", "z")):
        timestamp_text = timestamp_text[:-1] + "+00:00"
    # Handle expected failures with a clear, actionable message.
    try:
        parsed = datetime.fromisoformat(timestamp_text)
    except ValueError as exc:
        raise ValueError(f"Could not parse The Analyst lastUpdated value: {value!r}.") from exc

    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)


analyst_eligible = False
analyst_rows: list[dict[str, Any]] = []
analyst_last_updated: datetime | None = None
analyst_elapsed_seconds: float | None = None
analyst_age_days: float | None = None

# Validate the input before continuing with later processing.
if analyst_retrieval_error is not None:
    expected_data_metadata = {
        "included": False,
        "last_updated": None,
        "age_days": None,
        "reason": "Expected-performance data could not be retrieved or validated.",
    }
else:
    # Handle expected failures with a clear, actionable message.
    try:
        # Validate the input before continuing with later processing.
        if not isinstance(analyst_payload, dict):
            raise ValueError("The Analyst response must be a JSON object.")
        analyst_last_updated = parse_last_updated(analyst_payload.get("lastUpdated"))
        raw_analyst_rows = analyst_payload.get("data")
        # Validate the input before continuing with later processing.
        if not isinstance(raw_analyst_rows, list):
            raise ValueError("The Analyst response has no valid data list.")
        # Validate the input before continuing with later processing.
        if not all(isinstance(row, dict) for row in raw_analyst_rows):
            raise ValueError("Every The Analyst data entry must be a JSON object.")

        analyst_rows = raw_analyst_rows
        analyst_elapsed_seconds = (
            datetime.now(timezone.utc) - analyst_last_updated
        ).total_seconds()
        analyst_age_days = analyst_elapsed_seconds / 86400

        expected_data_metadata = {
            "included": False,
            "last_updated": analyst_last_updated.isoformat(),
            "age_days": round(analyst_age_days, 4),
        }

        # Choose the appropriate path for the current data state.
        if analyst_elapsed_seconds < 0:
            print(
                "Warning: The Analyst lastUpdated timestamp is in the future; "
                "the otherwise valid dataset remains eligible."
            )
            analyst_eligible = True
        # Choose the appropriate path for the current data state.
        elif analyst_elapsed_seconds <= MAX_EXPECTED_AGE_SECONDS:
            analyst_eligible = True
        else:
            expected_data_metadata["reason"] = (
                "Expected-performance dataset is more than 7 days old."
            )
            print("The Analyst dataset is stale; expected metrics will be excluded.")
    except Exception as exc:
        analyst_eligible = False
        analyst_rows = []
        expected_data_metadata = {
            "included": False,
            "last_updated": None,
            "age_days": None,
            "reason": "Expected-performance data could not be retrieved or validated.",
        }
        print(f"Warning: Analyst enrichment failed structural validation: {exc}")

print(f"Analyst data eligible for interactive matching: {analyst_eligible}")


The Analyst dataset is stale; expected metrics will be excluded.
Analyst data eligible for interactive matching: False


## 8. Rank candidates and confirm every proposed match

Candidate ranking is deterministic, but no pair is accepted automatically. Every proposed pair must receive an explicit `y`; after `n`, the next plausible unused candidate is offered.


In [10]:
# Set workflow configuration value: ANALYST_NAME_FIELDS.
ANALYST_NAME_FIELDS = (
    "contestantName",
    "contestantKnownName",
    "contestantShortName",
)


# Handle nonempty strings for reuse in the workflow.
def unique_nonempty_strings(values: list[Any]) -> list[str]:
    result: list[str] = []
    seen: set[str] = set()
    # Process each available item while preserving the current workflow state.
    for value in values:
        if isinstance(value, str) and value.strip() and value.strip() not in seen:
            cleaned = value.strip()
            result.append(cleaned)
            seen.add(cleaned)
    return result


# Handle names for reuse in the workflow.
def analyst_names(row: dict[str, Any]) -> list[str]:
    return unique_nonempty_strings([row.get(field) for field in ANALYST_NAME_FIELDS])


# Handle names for reuse in the workflow.
def source_names(team_id: int) -> list[str]:
    return unique_nonempty_strings(
        [teams_by_id[team_id]["team"], standings_by_id[team_id]["team"]]
    )


# Handle name similarity for reuse in the workflow.
def best_name_similarity(left_names: list[str], right_names: list[str]) -> float:
    if not left_names or not right_names:
        return 0.0
    return max(
        difflib.SequenceMatcher(None, normalize_name(left), normalize_name(right)).ratio()
        for left in left_names
        for right in right_names
    )


sofascore_code_counts = Counter(
    code for code in sofascore_codes.values() if code is not None
)
analyst_codes = [normalize_code(row.get("contestantCode")) for row in analyst_rows]
analyst_code_counts = Counter(code for code in analyst_codes if code is not None)


# Handle candidate for reuse in the workflow.
def rank_candidate(team_id: int, analyst_index: int) -> dict[str, Any] | None:
    row = analyst_rows[analyst_index]
    left_names = source_names(team_id)
    right_names = analyst_names(row)
    left_normalized = {normalize_name(name) for name in left_names}
    right_normalized = {normalize_name(name) for name in right_names}
    similarity = best_name_similarity(left_names, right_names)

    # Choose the appropriate path for the current data state.
    if left_normalized & right_normalized:
        tier, reason = 4, "normalized exact name"
    # Choose the appropriate path for the current data state.
    elif {canonical_name(name) for name in left_names} & {
        canonical_name(name) for name in right_names
    }:
        tier, reason = 3, "explicit alias"
    else:
        sofa_code = sofascore_codes.get(team_id)
        analyst_code = analyst_codes[analyst_index]
        unique_code_match = (
            sofa_code is not None
            and sofa_code == analyst_code
            and sofascore_code_counts[sofa_code] == 1
            and analyst_code_counts[analyst_code] == 1
        )
        # Choose the appropriate path for the current data state.
        if unique_code_match:
            tier, reason = 2, "unique team code"
        # Choose the appropriate path for the current data state.
        elif similarity >= SIMILARITY_PROMPT_THRESHOLD:
            tier, reason = 1, "name similarity"
        else:
            return None

    return {
        "analyst_index": analyst_index,
        "tier": tier,
        "reason": reason,
        "similarity": similarity,
    }


candidate_lists: dict[int, list[dict[str, Any]]] = {}
if analyst_eligible:
    # Process each available item while preserving the current workflow state.
    for team_id in standings_by_id:
        candidates = [
            candidate
            for analyst_index in range(len(analyst_rows))
            if (candidate := rank_candidate(team_id, analyst_index)) is not None
        ]
        candidate_lists[team_id] = sorted(
            candidates,
            key=lambda item: (-item["tier"], -item["similarity"], item["analyst_index"]),
        )


In [11]:
# Handle candidate evidence for reuse in the workflow.
def print_candidate_evidence(team_id: int, candidate: dict[str, Any]) -> None:
    row = analyst_rows[candidate["analyst_index"]]
    print("\nPotential team match detected:")
    print(f"SofaScore: {standings_by_id[team_id]['team']}")
    print(f"Local mapping: {teams_by_id[team_id]['team']}")
    print(f"The Analyst name: {row.get('contestantName')}")
    print(f"The Analyst known name: {row.get('contestantKnownName')}")
    print(f"The Analyst short name: {row.get('contestantShortName')}")
    print(f"SofaScore code: {sofascore_codes.get(team_id)}")
    print(f"The Analyst code: {normalize_code(row.get('contestantCode'))}")
    print(f"Proposal reason: {candidate['reason']}")
    print(f"Name similarity: {candidate['similarity']:.3f}")


# Handle pair confirmation for reuse in the workflow.
def request_pair_confirmation(team_id: int, candidate: dict[str, Any]) -> bool | None:
    pair_key = (team_id, candidate["analyst_index"])
    if pair_key in confirmed_name_matches:
        return confirmed_name_matches[pair_key]

    print_candidate_evidence(team_id, candidate)
    while True:
        # Handle expected failures with a clear, actionable message.
        try:
            response = input("Are these the same team? [y/n]: ").strip().lower()
        except EOFError:
            print("Warning: interactive input is unavailable; no unconfirmed pair will be accepted.")
            return None

        if response in {"y", "n"}:
            accepted = response == "y"
            confirmed_name_matches[pair_key] = accepted
            return accepted
        print("Please enter 'y' or 'n'.")


confirmed_name_matches: dict[tuple[int, int], bool] = {}
rejected_name_matches: set[tuple[int, int]] = set()
claimed_analyst_records: set[int] = set()
confirmed_matches: dict[int, int] = {}
match_proposals_shown = 0
interactive_input_available = True

if analyst_eligible:
    ordered_team_ids = sorted(
        standings_by_id, key=lambda team_id: standings_by_id[team_id]["position"]
    )
    # Process each available item while preserving the current workflow state.
    for team_id in ordered_team_ids:
        if not interactive_input_available:
            break

        # Process each available item while preserving the current workflow state.
        for candidate in candidate_lists.get(team_id, []):
            analyst_index = candidate["analyst_index"]
            pair_key = (team_id, analyst_index)
            if analyst_index in claimed_analyst_records or pair_key in rejected_name_matches:
                continue

            match_proposals_shown += 1
            accepted = request_pair_confirmation(team_id, candidate)
            if accepted is None:
                interactive_input_available = False
                break
            if accepted:
                confirmed_matches[team_id] = analyst_index
                claimed_analyst_records.add(analyst_index)
                break

            rejected_name_matches.add(pair_key)

print(f"User-confirmed matches: {len(confirmed_matches)}")
print(f"Rejected candidate pairs: {len(rejected_name_matches)}")


User-confirmed matches: 0
Rejected candidate pairs: 0


## 9. Merge confirmed expected metrics and calculate differences

A confirmed record must contain all four expected metrics. If any confirmed record is malformed, expected enrichment is discarded for the entire snapshot.


In [12]:
# Set workflow configuration value: EXPECTED_FIELDS.
EXPECTED_FIELDS = (
    "expected_position",
    "expected_points",
    "expected_goals_for",
    "expected_goals_against",
    "position_difference",
    "points_difference",
)

final_teams_by_id = {
    team_id: record.copy() for team_id, record in standings_by_id.items()
}
expected_metrics_by_team: dict[int, dict[str, Any]] = {}

# Validate the input before continuing with later processing.
if analyst_eligible and confirmed_matches:
    # Handle expected failures with a clear, actionable message.
    try:
        # Process each available item while preserving the current workflow state.
        for team_id, analyst_index in confirmed_matches.items():
            row = analyst_rows[analyst_index]
            x_position = row.get("xPos")
            x_points = row.get("xPts")
            x_goals_for = row.get("xG")
            x_goals_against = row.get("xGA")

            # Validate the input before continuing with later processing.
            if (
                not is_number(x_position)
                or not float(x_position).is_integer()
                or not 1 <= int(x_position) <= EXPECTED_TEAM_COUNT
            ):
                raise ValueError(
                    f"Confirmed Analyst record for team ID {team_id} has invalid xPos."
                )
            # Process each available item while preserving the current workflow state.
            for field_name, value in {
                "xPts": x_points,
                "xG": x_goals_for,
                "xGA": x_goals_against,
            }.items():
                # Validate the input before continuing with later processing.
                if not is_number(value) or value < 0:
                    raise ValueError(
                        f"Confirmed Analyst record for team ID {team_id} has invalid {field_name}."
                    )

            expected_metrics_by_team[team_id] = {
                "expected_position": int(x_position),
                "expected_points": float(x_points),
                "expected_goals_for": float(x_goals_for),
                "expected_goals_against": float(x_goals_against),
            }

        # Process each available item while preserving the current workflow state.
        for team_id, record in final_teams_by_id.items():
            if team_id not in expected_metrics_by_team:
                record.update({field: None for field in EXPECTED_FIELDS})
                continue

            metrics = expected_metrics_by_team[team_id]
            record.update(metrics)
            record["position_difference"] = (
                metrics["expected_position"] - record["position"]
            )
            record["points_difference"] = round(
                record["points"] - metrics["expected_points"], 2
            )

        expected_data_metadata["included"] = True
        expected_data_metadata.pop("reason", None)
    except Exception as exc:
        print(f"Warning: confirmed Analyst metrics are invalid; enrichment excluded: {exc}")
        expected_metrics_by_team = {}
        final_teams_by_id = {
            team_id: record.copy() for team_id, record in standings_by_id.items()
        }
        expected_data_metadata["included"] = False
        expected_data_metadata["reason"] = (
            "Expected-performance data could not be retrieved or validated."
        )
elif analyst_eligible:
    expected_data_metadata["included"] = False
    expected_data_metadata["reason"] = (
        "No Analyst team mappings were confirmed by the user."
    )


## 10. Build and validate the normalized output

Validation runs before the output file is opened, preventing serious structural inconsistencies from producing a misleading snapshot.


In [13]:
# Run this self-contained workflow step using the prepared inputs.
ordered_output_teams = sorted(
    final_teams_by_id.values(), key=lambda record: record["position"]
)
output_data = {
    "competition": COMPETITION,
    "season": SEASON,
    "captured_at": capture_datetime.isoformat(timespec="seconds"),
    "sources": {
        "standings": STANDINGS_URL,
        "expected_points": EXPECTED_POINTS_URL,
    },
    "expected_data": expected_data_metadata,
    "teams": {str(record["team_id"]): record for record in ordered_output_teams},
}


In [14]:
# Validate output for reuse in the workflow.
def validate_output() -> None:
    mapping_ids = set(teams_by_id)
    standings_ids = set(standings_by_id)
    output_ids = {record["team_id"] for record in output_data["teams"].values()}

    # Validate the input before continuing with later processing.
    if len(mapping_ids) != EXPECTED_TEAM_COUNT:
        raise ValueError(f"Authoritative mapping must contain {EXPECTED_TEAM_COUNT} teams.")
    # Validate the input before continuing with later processing.
    if mapping_ids != standings_ids or mapping_ids != output_ids:
        raise ValueError("Mapping, standings, and output team-ID sets must be identical.")
    # Validate the input before continuing with later processing.
    if len(output_data["teams"]) != EXPECTED_TEAM_COUNT:
        raise ValueError(f"Output must contain exactly {EXPECTED_TEAM_COUNT} teams.")

    positions: list[int] = []
    # Process each available item while preserving the current workflow state.
    for team_key, record in output_data["teams"].items():
        team_id = record.get("team_id")
        # Validate the input before continuing with later processing.
        if team_key != str(team_id):
            raise ValueError(f"Output key {team_key!r} does not match team_id {team_id!r}.")
        # Validate the input before continuing with later processing.
        if not isinstance(record.get("team"), str) or not record["team"].strip():
            raise ValueError(f"Output team {team_id} has an invalid display name.")

        positions.append(require_integer(record.get("position"), f"team {team_id} position", minimum=1))
        # Process each available item while preserving the current workflow state.
        for field in ("matches", "wins", "draws", "losses", "goals_for", "goals_against"):
            require_integer(record.get(field), f"team {team_id} {field}", minimum=0)
        # Validate the input before continuing with later processing.
        if not is_number(record.get("points")):
            raise ValueError(f"Output team {team_id} has non-numeric points.")
        # Validate the input before continuing with later processing.
        if record["matches"] != record["wins"] + record["draws"] + record["losses"]:
            raise ValueError(f"Match-result arithmetic failed for team {team_id}.")
        # Validate the input before continuing with later processing.
        if record["goal_difference"] != record["goals_for"] - record["goals_against"]:
            raise ValueError(f"Goal-difference arithmetic failed for team {team_id}.")

    # Validate the input before continuing with later processing.
    if set(positions) != set(range(1, EXPECTED_TEAM_COUNT + 1)) or len(positions) != len(set(positions)):
        raise ValueError("Standings positions must be unique and cover 1 through 18.")

    expected_included = output_data["expected_data"].get("included")
    # Validate the input before continuing with later processing.
    if expected_included:
        # Validate the input before continuing with later processing.
        if analyst_elapsed_seconds is None or analyst_elapsed_seconds > MAX_EXPECTED_AGE_SECONDS:
            raise ValueError("Included Analyst data does not satisfy the freshness rule.")
        # Validate the input before continuing with later processing.
        if not confirmed_matches or not expected_metrics_by_team:
            raise ValueError("Included Analyst data requires confirmed, validated mappings.")
        # Validate the input before continuing with later processing.
        if len(set(confirmed_matches.values())) != len(confirmed_matches):
            raise ValueError("An Analyst record was assigned to multiple Bundesliga teams.")

        # Process each available item while preserving the current workflow state.
        for team_id, record in final_teams_by_id.items():
            # Validate the input before continuing with later processing.
            if team_id in confirmed_matches:
                pair_key = (team_id, confirmed_matches[team_id])
                # Validate the input before continuing with later processing.
                if confirmed_name_matches.get(pair_key) is not True:
                    raise ValueError(f"Team {team_id} was enriched without explicit y confirmation.")
                # Validate the input before continuing with later processing.
                if pair_key in rejected_name_matches:
                    raise ValueError(f"Rejected pair {pair_key} was used in output.")
                metrics = expected_metrics_by_team[team_id]
                # Process each available item while preserving the current workflow state.
                for field, expected_value in metrics.items():
                    # Validate the input before continuing with later processing.
                    if not math.isclose(record[field], expected_value, rel_tol=0, abs_tol=1e-9):
                        raise ValueError(f"Expected metric {field} is incorrect for team {team_id}.")
                expected_position_difference = metrics["expected_position"] - record["position"]
                expected_points_difference = round(record["points"] - metrics["expected_points"], 2)
                # Validate the input before continuing with later processing.
                if record["position_difference"] != expected_position_difference:
                    raise ValueError(f"Position difference is incorrect for team {team_id}.")
                # Validate the input before continuing with later processing.
                if not math.isclose(
                    record["points_difference"], expected_points_difference, rel_tol=0, abs_tol=1e-9
                ):
                    raise ValueError(f"Points difference is incorrect for team {team_id}.")
            else:
                missing_expected_fields = [
                    field for field in EXPECTED_FIELDS if field not in record
                ]
                # Validate the input before continuing with later processing.
                if missing_expected_fields:
                    raise ValueError(
                        f"Unmatched team {team_id} is missing expected fields: "
                        f"{missing_expected_fields}."
                    )
                # Validate the input before continuing with later processing.
                if any(record[field] is not None for field in EXPECTED_FIELDS):
                    raise ValueError(f"Unmatched team {team_id} must have null expected fields.")
    else:
        # Validate the input before continuing with later processing.
        if not output_data["expected_data"].get("reason"):
            raise ValueError("Excluded Analyst metadata must contain a reason.")
        # Process each available item while preserving the current workflow state.
        for team_id, record in final_teams_by_id.items():
            unexpected_fields = [field for field in EXPECTED_FIELDS if field in record]
            # Validate the input before continuing with later processing.
            if unexpected_fields:
                raise ValueError(
                    f"Excluded Analyst data left expected fields on team {team_id}: {unexpected_fields}."
                )


validate_output()
print("Normalized output passed all validation checks.")


Normalized output passed all validation checks.


## 11. Save the JSON snapshot

The validated object is written as readable UTF-8 JSON in the configured project folder.


In [15]:
filename_timestamp = capture_datetime.strftime("%Y-%m-%d_%H-%M-%S")
output_path = ensure_directory(DERIVED_BUNDESLIGA_SNAPSHOTS_DIR) / f"bundesliga_table_{filename_timestamp}.json"

# Use the resource only within this controlled scope.
with output_path.open("w", encoding="utf-8", newline="\n") as file:
    json.dump(
        output_data,
        file,
        indent=4,
        ensure_ascii=False,
    )
    file.write("\n")

print(f"Saved snapshot: {output_path}")


Saved snapshot: C:\kickbase project\outputs\derived\bundesliga_snapshots\bundesliga_table_2026-08-23_00-49-56.json


## 12. Execution summary and inspection table

The DataFrame is for interactive inspection only; the JSON file remains the authoritative generated artifact.


In [16]:
# Run this self-contained workflow step using the prepared inputs.
unmatched_team_ids = sorted(
    set(standings_by_id) - set(confirmed_matches),
    key=lambda team_id: standings_by_id[team_id]["position"],
)
ignored_analyst_indices = sorted(set(range(len(analyst_rows))) - claimed_analyst_records)
unmatched_team_names = [standings_by_id[team_id]["team"] for team_id in unmatched_team_ids]
ignored_analyst_names = [
    analyst_rows[index].get("contestantName", f"record {index}")
    for index in ignored_analyst_indices
]
rejected_pair_labels = [
    (
        standings_by_id[team_id]["team"],
        analyst_rows[analyst_index].get("contestantName", f"record {analyst_index}"),
    )
    for team_id, analyst_index in sorted(rejected_name_matches)
]

print("\nBundesliga snapshot complete\n")
print(f"Teams captured: {len(output_data['teams'])}")
print(f"Output file: {output_path.name}\n")
print("SofaScore standings: retrieved successfully\n")
print("Expected-performance data:")
print(f"Included: {'Yes' if expected_data_metadata['included'] else 'No'}")
print(f"Last updated: {expected_data_metadata.get('last_updated')}")
print(f"Age: {expected_data_metadata.get('age_days')} days")
if expected_data_metadata.get("reason"):
    print(f"Reason: {expected_data_metadata['reason']}")
print("\nAutomatically finalized matches: 0")
print(f"Match proposals shown: {match_proposals_shown}")
print(f"User-confirmed matches: {len(confirmed_matches)}")
print(f"Rejected pairs: {rejected_pair_labels or 'None'}")
print(f"Unmatched Bundesliga teams: {unmatched_team_names or 'None'}")
print(f"Ignored Analyst records: {ignored_analyst_names or 'None'}")

inspection_df = (
    pd.DataFrame(output_data["teams"].values())
    .sort_values("position")
    .reset_index(drop=True)
)
display(inspection_df)



Bundesliga snapshot complete

Teams captured: 18
Output file: bundesliga_table_2026-08-23_00-49-56.json

SofaScore standings: retrieved successfully

Expected-performance data:
Included: No
Last updated: 2026-05-26T06:13:31.164086+00:00
Age: 88.692 days
Reason: Expected-performance dataset is more than 7 days old.

Automatically finalized matches: 0
Match proposals shown: 0
User-confirmed matches: 0
Rejected pairs: None
Unmatched Bundesliga teams: ['1. FC Köln', 'Bayer 04 Leverkusen', 'FC Bayern München', 'Borussia Dortmund', "Borussia M'gladbach", 'Eintracht Frankfurt', 'FC Augsburg', '1. FSV Mainz 05', 'Hamburger SV', 'RB Leipzig', 'SC Freiburg', 'SC Paderborn 07', 'FC Schalke 04', 'SV 07 Elversberg', 'TSG Hoffenheim', '1. FC Union Berlin', 'VfB Stuttgart', 'SV Werder Bremen']
Ignored Analyst records: ['1. FSV Mainz 05', 'VfB Stuttgart 1893', '1. FC Heidenheim 1846', 'TSG 1899 Hoffenheim', 'SC Freiburg', 'Hamburger SV', 'Bayer 04 Leverkusen', 'FC St. Pauli', '1. FC Köln', 'RasenBall

,team_id,team,position,matches,wins,draws,losses,goals_for,goals_against,goal_difference,points
0,2671,1. FC Köln,1,0,0,0,0,0,0,0,0
1,2681,Bayer 04 Leverkusen,2,0,0,0,0,0,0,0,0
2,2672,FC Bayern München,3,0,0,0,0,0,0,0,0
3,2673,Borussia Dortmund,4,0,0,0,0,0,0,0,0
4,2527,Borussia M'gladbach,5,0,0,0,0,0,0,0,0
5,2674,Eintracht Frankfurt,6,0,0,0,0,0,0,0,0
6,2600,FC Augsburg,7,0,0,0,0,0,0,0,0
7,2556,1. FSV Mainz 05,8,0,0,0,0,0,0,0,0
8,2676,Hamburger SV,9,0,0,0,0,0,0,0,0
9,36360,RB Leipzig,10,0,0,0,0,0,0,0,0
